# Sesión 4 · Notebook previo
## Modelos fundacionales para series de tiempo frente a un baseline clásico

**Curso:** Ciencia de datos aumentada con IA: del dato a la decisión
**Tiempo estimado:** 45 a 55 minutos · **Entrega:** este notebook ejecutado, antes de la clase

### Qué vas a hacer
1. Pronosticar una serie de tiempo (tickets diarios de TiendaTech) con un modelo clásico, **ARIMA**, como baseline.
2. Pronosticar la misma serie con **Chronos**, un modelo fundacional preentrenado para series de tiempo, en modo **zero-shot** (sin entrenarlo con tus datos).
3. Comparar ambos con una métrica de error y con el costo de tiempo de cómputo.
4. Dejar preparada la comparación de un modelo fundacional tabular (TabPFN) frente a un modelo clásico, que se usará en el taller.

**Nota:** las celdas de Chronos y TabPFN instalan librerías basadas en PyTorch, que pueden tardar varios minutos la primera vez en Colab. Ejecuta este notebook con tiempo de sobra.

## 0. Cargar los datos

In [1]:
!pip install -q pandas numpy matplotlib statsmodels scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Sube serie_tickets_diarios.csv a tu sesión de Colab antes de ejecutar.
df = pd.read_csv("serie_tickets_diarios.csv", parse_dates=["fecha"])
df = df.set_index("fecha").asfreq("D")
print(df.shape)
df.plot(y="tickets_diarios", figsize=(10, 3), title="Tickets diarios de TiendaTech")
plt.show()

In [ ]:
HORIZONTE = 14  # días a pronosticar
serie_entrenamiento = df["tickets_diarios"].iloc[:-HORIZONTE]
serie_prueba = df["tickets_diarios"].iloc[-HORIZONTE:]
print(f"Entrenamiento: {len(serie_entrenamiento)} días | Prueba: {len(serie_prueba)} días")

## 1. Baseline clásico: ARIMA
ARIMA necesita que tú (o una búsqueda automática) elijas su orden y que el modelo se **entrene específicamente con esta serie**. Es el punto de comparación justo: un modelo clásico, bien ajustado, entrenado con los datos del caso.

In [ ]:
import time
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

t0 = time.time()
modelo_arima = ARIMA(serie_entrenamiento, order=(2, 1, 2)).fit()
pronostico_arima = modelo_arima.forecast(steps=HORIZONTE)
tiempo_arima = time.time() - t0

mae_arima = mean_absolute_error(serie_prueba, pronostico_arima)
mape_arima = mean_absolute_percentage_error(serie_prueba, pronostico_arima)
print(f"ARIMA(2,1,2) -> MAE: {mae_arima:.2f} | MAPE: {mape_arima:.1%} | tiempo: {tiempo_arima:.2f} s")

### TU TURNO 1
Prueba **otros dos órdenes de ARIMA** (por ejemplo, (1,1,1) y (3,1,1)) y anota cuál da mejor MAE. En la práctica, esta búsqueda se automatiza (por ejemplo, con `pmdarima.auto_arima`), pero hacerlo a mano una vez ayuda a entender qué está pasando.

In [ ]:
for orden in [(1,1,1), (3,1,1)]:
    m = ARIMA(serie_entrenamiento, order=orden).fit()
    p = m.forecast(steps=HORIZONTE)
    print(orden, "MAE:", round(mean_absolute_error(serie_prueba, p), 2))

respuesta_1 = """
Escribe aquí qué orden funcionó mejor y qué tan sensible fue el resultado al cambiar el orden.
"""

## 2. Modelo fundacional: Chronos (zero-shot)
[Chronos](https://arxiv.org/abs/2403.07815) es un modelo de lenguaje entrenado sobre series de tiempo tokenizadas: convierte los valores numéricos en "tokens" y aprende a predecir la continuación, igual que un LLM predice la siguiente palabra. Lo interesante es que se puede usar **sin entrenarlo con tu serie** (zero-shot): nunca vio los tickets de TiendaTech y aun así puede pronosticar.

In [11]:
!pip install -q chronos-forecasting torch --extra-index-url https://download.pytorch.org/whl/cpu

In [ ]:
import time
import torch
from chronos import ChronosPipeline
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

t0 = time.time()
pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",   # también existen versiones tiny, base, large: a mayor tamaño, más lento y (en general) más preciso
    device_map="cpu",
    torch_dtype=torch.float32,
)
tiempo_carga_chronos = time.time() - t0
print(f"Modelo cargado en {tiempo_carga_chronos:.1f} s")

contexto = torch.tensor(serie_entrenamiento.values, dtype=torch.float32)
t0 = time.time()
# Se pasa 'contexto' como argumento posicional para evitar problemas con nombres de argumentos según la versión de la librería
pronostico_muestras = pipeline.predict(contexto, prediction_length=HORIZONTE, num_samples=20)
tiempo_chronos = time.time() - t0

# Chronos devuelve varias trayectorias posibles (es un modelo probabilístico); usamos la mediana como pronóstico puntual
pronostico_chronos = np.median(pronostico_muestras[0].numpy(), axis=0)

mae_chronos = mean_absolute_error(serie_prueba, pronostico_chronos)
mape_chronos = mean_absolute_percentage_error(serie_prueba, pronostico_chronos)
print(f"Chronos (zero-shot) -> MAE: {mae_chronos:.2f} | MAPE: {mape_chronos:.1%} | tiempo de inferencia: {tiempo_chronos:.2f} s")

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(serie_entrenamiento.index[-30:], serie_entrenamiento.values[-30:], label="Histórico")
plt.plot(serie_prueba.index, serie_prueba.values, label="Real", marker="o")
plt.plot(serie_prueba.index, pronostico_arima, label="ARIMA", linestyle="--")
plt.plot(serie_prueba.index, pronostico_chronos, label="Chronos (zero-shot)", linestyle="--")
plt.legend(); plt.title("Comparación de pronósticos"); plt.show()

pd.DataFrame({
    "modelo": ["ARIMA(2,1,2)", "Chronos zero-shot"],
    "mae": [mae_arima, mae_chronos],
    "mape": [mape_arima, mape_chronos],
    "tiempo_inferencia_s": [tiempo_arima, tiempo_chronos],
    "requirio_entrenamiento_con_estos_datos": ["Sí", "No"],
})

### TU TURNO 2
1. ¿Qué modelo ganó en error (MAE/MAPE)? ¿Y en tiempo de cómputo?
2. ARIMA se entrenó específicamente con esta serie; Chronos nunca la había visto. Teniendo eso en cuenta, ¿qué tan sorprendente te parece el resultado?
3. ¿En qué escenario de tu proyecto preferirías un modelo zero-shot como Chronos en vez de entrenar un modelo específico? Piensa en casos con pocos datos históricos o muchas series distintas.

In [14]:
respuesta_2 = """
Escribe aquí tus respuestas.
"""

## 3. Preparación para el taller: TabPFN frente a un modelo clásico
[TabPFN](https://www.nature.com/articles/s41586-024-08328-6) es un modelo fundacional, pero para **datos tabulares** (no series de tiempo): predice en segundos sobre datasets pequeños, sin que tú ajustes hiperparámetros. En el taller lo compararán contra un modelo clásico (gradient boosting) sobre una tarea derivada de tickets_soporte.csv: predecir si un ticket tendrá satisfacción baja (1-2) a partir de sus demás columnas.

Aquí dejamos listo el baseline clásico para que en el taller solo agreguen la comparación con TabPFN.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder

tickets = pd.read_csv("tickets_soporte.csv")
tickets = tickets.dropna(subset=["satisfaccion"]).copy()
tickets["satisfaccion_baja"] = (tickets["satisfaccion"] <= 2).astype(int)

caracteristicas = ["canal", "categoria", "horas_resolucion"]
X = tickets[caracteristicas].copy()
for col in ["canal", "categoria"]:
    X[col] = LabelEncoder().fit_transform(X[col])
y = tickets["satisfaccion_baja"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1, stratify=y)
print(f"Entrenamiento: {len(X_train)} filas | Prueba: {len(X_test)} filas | Positivos en prueba: {y_test.sum()}")

t0 = time.time()
clf_clasico = GradientBoostingClassifier(random_state=1).fit(X_train, y_train)
tiempo_clasico = time.time() - t0
pred_clasico = clf_clasico.predict(X_test)
print(f"Gradient boosting -> accuracy: {accuracy_score(y_test, pred_clasico):.2f} | F1: {f1_score(y_test, pred_clasico):.2f} | tiempo: {tiempo_clasico:.2f} s")

En el taller instalarán `tabpfn` y compararán: `from tabpfn import TabPFNClassifier; clf = TabPFNClassifier(); clf.fit(X_train, y_train)`. Revisa la documentación vigente de TabPFN antes del taller, porque su interfaz puede haber cambiado desde que se escribió este notebook.

## 4. Cierre
### TU TURNO 3 · Reflexión final (máx. 5 líneas)
Con lo visto en Chronos, ¿qué ventaja concreta le ves a un modelo fundacional zero-shot frente a entrenar un modelo por cada serie o cada dataset pequeño? ¿Qué desventaja o riesgo le ves?

In [17]:
respuesta_3 = """
Escribe aquí tu reflexión.
"""

DECLARACION_IAG = """
Nivel de uso de IAG: (1 a 5, según el acuerdo del curso)
Herramientas usadas y para qué:
Cómo verifiqué la calidad y veracidad del contenido generado:
"""

### Antes de entregar
- Sube `serie_tickets_diarios.csv` y `tickets_soporte.csv` a tu sesión de Colab antes de ejecutar.
- Si la instalación de `chronos-forecasting` falla o tarda demasiado, prueba con el modelo `amazon/chronos-t5-tiny` (más liviano) o consulta al profesor.
- Ejecuta **Entorno de ejecución → Reiniciar y ejecutar todo** y verifica que no haya errores.
- Comprueba que las celdas `respuesta_1` a `respuesta_3` y `DECLARACION_IAG` estén completas.